In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.model_selection import train_test_split

#### przygotowanie danych do testu Azure 

## wczytanie danych (jak w fazie treningu)
df = joblib.load('df_with_all_clusters.joblib')
umap_embeddings = np.load('umap_embeddings_10d.npy')

## odtworzenie X i y
X = umap_embeddings
y = df['nmf_topic'] #### to były etykiety treningowe

## odtworzenie podziału na zbiór treningowy i testowy
#### random_state=42 i stratify=y dokładnie tak samo jak przy treningu
#### dodatkowo dzielenie indeksów (df.index) żeby potem odnaleźć tytuły
print("Odtwarzanie podziału train/test...")
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

## losowanie 5 próbek ze zbioru testowego
print("Losowanie 5 próbek...")
#### 5 losowych indeksów z tablicy X_test
random_indices = np.random.choice(len(X_test), 5, replace=False)

## wyciąganie danych dla tych indeksów
sample_X = X_test[random_indices]       # wektory (to do wysłania do modelu do Azure)
sample_y = y_test.iloc[random_indices]  # prawdziwe ID tematu
sample_titles = df.loc[idx_test[random_indices], 'title'] # tytuły artykułów

## przygotowanie JSON do wysyłki
##### Azure oczekuje formatu: {"data": [[...], [...]]}
##### trzeba zamienić numpy array na listę
payload_data = {"data": sample_X.tolist()}
json_payload = json.dumps(payload_data)

## wyświetlenie ściągi
print("\n---> ŚCIĄGAWKA DO TESTU (Oczekiwane Wyniki)")
expected_results = []
for i, (title, true_topic) in enumerate(zip(sample_titles, sample_y)):
    print(f"\nArtykuł {i+1}:")
    print(f"Tytuł: {title}")
    print(f"Prawdziwy Temat NMF: {true_topic}")
    expected_results.append(true_topic)

print("\nGotowy payload JSON (pierwsze 100 znaków):")
print(str(json_payload)[:100] + "...")

Odtwarzanie podziału train/test...
Losowanie 5 próbek...

---> ŚCIĄGAWKA DO TESTU (Oczekiwane Wyniki)

Artykuł 1:
Tytuł: Letter to the Editor: ”Role of the Neurosurgeon in Times of Coronavirus Disease 2019 (COVID-19): The Importance of Focus in Critical Care”
Prawdziwy Temat NMF: 7

Artykuł 2:
Tytuł: Lower CMV and EBV Exposure in Children With Kawasaki Disease Suggests an Under-Challenged Immune System
Prawdziwy Temat NMF: 10

Artykuł 3:
Tytuł: Candidate new rotavirus species in Schreiber's bats, Serbia
Prawdziwy Temat NMF: 12

Artykuł 4:
Tytuł: Disparities in COVID-19 Information Sources and Knowledge in South Korea
Prawdziwy Temat NMF: 3

Artykuł 5:
Tytuł: Vigilância em saúde da COVID-19 no Brasil: busca de contatos nas comunidades como estratégia de autoproteção (preprint)/ pt
Prawdziwy Temat NMF: 11

Gotowy payload JSON (pierwsze 100 znaków):
{"data": [[6.534129619598389, 6.573187351226807, 4.614930629730225, 3.3724539279937744, 4.0486288070...


In [3]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient


### połączenie z workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id="2e6f9eb9-9633-4b76-949e-d54bf09060ba",
    resource_group_name="mai-ml-projects-eus",
    workspace_name="ml-project-covid19"
)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [5]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

### definicja endpointu
endpoint = ManagedOnlineEndpoint(
    name="covid19-lightgbm-endpoint",
    description="Endpoint dla modelu LightGBM COVID19",
    auth_mode="key"
)

ml_client.begin_create_or_update(endpoint).result()

ManagedOnlineEndpoint({'public_network_access': 'Enabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://covid19-lightgbm-endpoint.eastus.inference.ml.azure.com/score', 'openapi_uri': 'https://covid19-lightgbm-endpoint.eastus.inference.ml.azure.com/swagger.json', 'name': 'covid19-lightgbm-endpoint', 'description': 'Endpoint dla modelu LightGBM COVID19', 'tags': {}, 'properties': {'createdBy': 'Grzegorz Zieliński', 'createdAt': '2025-11-23T22:11:22.413608+0000', 'lastModifiedAt': '2025-11-23T22:11:22.413608+0000', 'azureml.onlineendpointid': '/subscriptions/2e6f9eb9-9633-4b76-949e-d54bf09060ba/resourcegroups/mai-ml-projects-eus/providers/microsoft.machinelearningservices/workspaces/ml-project-covid19/onlineendpoints/covid19-lightgbm-endpoint', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/2e6f9eb9-9633-4b76-949e-d54bf09060ba/providers/Microsoft.MachineLearningServices/locations/eastus/mfeOperationsStatus/oeidp:6f8445aa-fc72-4c43-a457-1f942c455154:ce0

In [7]:
### definicja deploymentu (powiązanie z modelem)
from azure.ai.ml.entities import ManagedOnlineDeployment, Environment, CodeConfiguration, BuildContext

deployment = ManagedOnlineDeployment(
    name="covid19-deployment",
    endpoint_name="covid19-lightgbm-endpoint",
    model="lgb_gbdt_best_model:1",  ## nazwa + wersja modelu w workspace
    environment="lgbm-env:1",
    code_configuration=CodeConfiguration(
        code="./covid19_deploy",          ### folder z score.py i lgbm_env.yml
        scoring_script="score.py"  ### plik scoringowy
    ),
    instance_type="Standard_DS2_v2",
    instance_count=1
)

ml_client.begin_create_or_update(deployment).result()

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint covid19-lightgbm-endpoint exists


......................................................................................................................................................................

ManagedOnlineDeployment({'private_network_connection': None, 'package_model': False, 'provisioning_state': 'Succeeded', 'endpoint_name': 'covid19-lightgbm-endpoint', 'type': 'Managed', 'name': 'covid19-deployment', 'description': None, 'tags': {}, 'properties': {'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/2e6f9eb9-9633-4b76-949e-d54bf09060ba/providers/Microsoft.MachineLearningServices/locations/eastus/mfeOperationsStatus/odidp:6f8445aa-fc72-4c43-a457-1f942c455154:ca3dee3f-4f09-470c-9fa0-8bf0dbde0b9b?api-version=2023-04-01-preview'}, 'print_as_yaml': False, 'id': '/subscriptions/2e6f9eb9-9633-4b76-949e-d54bf09060ba/resourceGroups/mai-ml-projects-eus/providers/Microsoft.MachineLearningServices/workspaces/ml-project-covid19/onlineEndpoints/covid19-lightgbm-endpoint/deployments/covid19-deployment', 'Resource__source_path': '', 'base_path': 'O:\\Projects\\covid19', 'creation_context': <azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.SystemData object

In [11]:
import requests
import json

### testowanie wdrożonego endpointu

print("Pobieranie danych dostępowych do endpointu...")

## pobranie adresu URL (Scoring URI)
endpoint_name = "covid19-lightgbm-endpoint"
endpoint_info = ml_client.online_endpoints.get(name=endpoint_name)
scoring_uri = endpoint_info.scoring_uri
print(f"Scoring URI: {scoring_uri}")

## pobranie klucza autoryzacyjnego (Primary Key)
keys = ml_client.online_endpoints.get_keys(name=endpoint_name)
auth_key = keys.primary_key
print("Klucz autoryzacyjny pobrany.")

## wysłanie żądania do Azure
print(f"\nWysyłanie 5 próbek do modelu w chmurze...")

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {auth_key}",
    "azureml-model-deployment": "covid19-deployment" #### wskazanie konkretnego wdrożenia
}

## użycie json_payload przygotowanego wcześniej
response = requests.post(scoring_uri, data=json_payload, headers=headers)

## wyświetlenie i interpretacja wyników
if response.status_code == 200:
    print("\nSukces! Otrzymano odpowiedź z chmury:")
    result = response.json()

    #### sprawdzenie i naprawa formatu
    if isinstance(result, str):
        print("(Info: Odpowiedź przyszła jako tekst, parsuję ponownie...)")
        result = json.loads(result)
    
    predictions = result["predictions"]
    
    print(f"Predykcje modelu: {predictions}")
    print("-" * 20)
    
    #### porównanie ze ściągawką (oczekiwanymi wynikami)
    print("Porównanie z rzeczywistością:")
    correct_count = 0
    for i, (pred, true_val, title) in enumerate(zip(predictions, expected_results, sample_titles)):
        is_correct = "TAK" if pred == true_val else "NIE"
        if pred == true_val: correct_count += 1
        
        print(f"\nArtykuł {i+1}:")
        print(f"Tytuł: {title[:100]}...") ##### skrócenie tytułu dla czytelności
        print(f"Prawdziwy Temat: {true_val} | Predykcja Azure: {pred} -> Poprawnie? {is_correct}")
        
    print("-" * 20)
    print(f"Skuteczność na próbce testowej: {correct_count}/5 ({correct_count/5*100}%)")
    
else:
    print(f"\nBŁĄD! Kod statusu: {response.status_code}")
    print(f"Treść błędu: {response.text}")

Pobieranie danych dostępowych do endpointu...
Scoring URI: https://covid19-lightgbm-endpoint.eastus.inference.ml.azure.com/score
Klucz autoryzacyjny pobrany.

Wysyłanie 5 próbek do modelu w chmurze...

Sukces! Otrzymano odpowiedź z chmury:
(Info: Odpowiedź przyszła jako tekst, parsuję ponownie...)
Predykcje modelu: [7, 9, 12, 1, 11]
--------------------
Porównanie z rzeczywistością:

Artykuł 1:
Tytuł: Letter to the Editor: ”Role of the Neurosurgeon in Times of Coronavirus Disease 2019 (COVID-19): The...
Prawdziwy Temat: 7 | Predykcja Azure: 7 -> Poprawnie? TAK

Artykuł 2:
Tytuł: Lower CMV and EBV Exposure in Children With Kawasaki Disease Suggests an Under-Challenged Immune Sys...
Prawdziwy Temat: 10 | Predykcja Azure: 9 -> Poprawnie? NIE

Artykuł 3:
Tytuł: Candidate new rotavirus species in Schreiber's bats, Serbia...
Prawdziwy Temat: 12 | Predykcja Azure: 12 -> Poprawnie? TAK

Artykuł 4:
Tytuł: Disparities in COVID-19 Information Sources and Knowledge in South Korea...
Prawdziwy Tema

In [13]:
##### Test 10 próbek

### przygotowanie Danych do Testu Azure (10 próbek)
print("Losowanie 10 próbek...")
random_indices = np.random.choice(len(X_test), 10, replace=False)

sample_X = X_test[random_indices]
sample_y = y_test.iloc[random_indices]
sample_titles = df.loc[idx_test[random_indices], 'title']

payload_data = {"data": sample_X.tolist()}
json_payload = json.dumps(payload_data)

print("\n---> ŚCIĄGAWKA (Oczekiwane Wyniki) ===")
expected_results = []
for i, (title, true_topic) in enumerate(zip(sample_titles, sample_y)):
    print(f"Art {i+1}: Temat {true_topic} -> {title[:50]}...")
    expected_results.append(true_topic)

Losowanie 10 próbek...

---> ŚCIĄGAWKA (Oczekiwane Wyniki) ===
Art 1: Temat 4 -> Motors of COVID-19 Vaccination Acceptance Scale (M...
Art 2: Temat 11 -> Plano Municipal de contingência do novo Coronavíru...
Art 3: Temat 12 -> Minimizing exposure to respiratory droplets, 'jet ...
Art 4: Temat 8 -> Genetic Variation, Diet, Inflammation, and the Ris...
Art 5: Temat 0 -> Is There Benefit from Stenting on Cognitive Functi...
Art 6: Temat 16 -> Delayed positive association between air temperatu...
Art 7: Temat 2 -> The first three waves of the Covid-19 pandemic hin...
Art 8: Temat 13 -> Evaluation of Antibody Response in Symptomatic and...
Art 9: Temat 5 -> The effects of the COVID-19 pandemic and quarantin...
Art 10: Temat 12 -> Etiology of severe community-acquired pneumonia du...


In [17]:
### testowanie wdrożonego endpointu - 10 próbek

print("Pobieranie danych dostępowych do endpointu...")

## pobranie adresu URL (Scoring URI)
endpoint_name = "covid19-lightgbm-endpoint"
endpoint_info = ml_client.online_endpoints.get(name=endpoint_name)
scoring_uri = endpoint_info.scoring_uri
print(f"Scoring URI: {scoring_uri}")

## pobranie klucza autoryzacyjnego (Primary Key)
keys = ml_client.online_endpoints.get_keys(name=endpoint_name)
auth_key = keys.primary_key
print("Klucz autoryzacyjny pobrany.")

## wysłanie żądania do Azure
print(f"\nWysyłanie 10 próbek do modelu w chmurze...")

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {auth_key}",
    "azureml-model-deployment": "covid19-deployment" #### wskazanie konkretnego wdrożenia
}

## użycie json_payload przygotowanego wcześniej
response = requests.post(scoring_uri, data=json_payload, headers=headers)

## wyświetlenie i interpretacja wyników
if response.status_code == 200:
    print("\nSukces! Otrzymano odpowiedź z chmury:")
    result = response.json()

    #### sprawdzenie i naprawa formatu
    if isinstance(result, str):
        print("(Info: Odpowiedź przyszła jako tekst, parsuję ponownie...)")
        result = json.loads(result)
    
    predictions = result["predictions"]
    
    print(f"Predykcje modelu: {predictions}")
    print("-" * 20)
    
    #### porównanie ze ściągawką (oczekiwanymi wynikami)
    print("Porównanie z rzeczywistością:")
    correct_count = 0
    for i, (pred, true_val, title) in enumerate(zip(predictions, expected_results, sample_titles)):
        is_correct = "TAK" if pred == true_val else "NIE"
        if pred == true_val: correct_count += 1
        
        print(f"\nArtykuł {i+1}:")
        print(f"Tytuł: {title[:100]}...") ##### skrócenie tytułu dla czytelności
        print(f"Prawdziwy Temat: {true_val} | Predykcja Azure: {pred} -> Poprawnie? {is_correct}")
        
    print("-" * 20)
    print(f"Skuteczność na próbce testowej: {correct_count}/10 ({correct_count/10*100}%)")
    
else:
    print(f"\nBŁĄD! Kod statusu: {response.status_code}")
    print(f"Treść błędu: {response.text}")

Pobieranie danych dostępowych do endpointu...
Scoring URI: https://covid19-lightgbm-endpoint.eastus.inference.ml.azure.com/score
Klucz autoryzacyjny pobrany.

Wysyłanie 10 próbek do modelu w chmurze...

Sukces! Otrzymano odpowiedź z chmury:
(Info: Odpowiedź przyszła jako tekst, parsuję ponownie...)
Predykcje modelu: [4, 11, 9, 16, 16, 16, 2, 13, 16, 12]
--------------------
Porównanie z rzeczywistością:

Artykuł 1:
Tytuł: Motors of COVID-19 Vaccination Acceptance Scale (MoVac-COVID19S): Evidence of Measurement Invariance...
Prawdziwy Temat: 4 | Predykcja Azure: 4 -> Poprawnie? TAK

Artykuł 2:
Tytuł: Plano Municipal de contingência do novo Coronavírus (covid-19) do município de Sítio Novo do Tocanti...
Prawdziwy Temat: 11 | Predykcja Azure: 11 -> Poprawnie? TAK

Artykuł 3:
Tytuł: Minimizing exposure to respiratory droplets, 'jet riders' and aerosols in air-conditioned hospital r...
Prawdziwy Temat: 12 | Predykcja Azure: 9 -> Poprawnie? NIE

Artykuł 4:
Tytuł: Genetic Variation, Diet, Inf